# OSM Landcover Classification - Example Usage

This notebook demonstrates how to use the `OSM_auto_label` package to:
1. Load and classify OSM polygon data
2. Visualize results on an interactive Leaflet map
3. Customize classification parameters

## Configuration

Set your input/output paths here:

In [ ]:
# =============================================================================
# USER CONFIGURATION - Modify these paths for your data
# =============================================================================

# Input shapefile containing OSM polygon data with 'landuse' and/or 'natural' columns
INPUT_SHAPEFILE = "path/to/your/osm_landuse.shp"

# Output shapefile for classified data
OUTPUT_SHAPEFILE = "output/classified_landcover.shp"

# Output HTML map
OUTPUT_MAP = "output/landcover_map.html"

# =============================================================================
# Optional: Run clustering analysis for exploration
RUN_CLUSTERING = False

## Setup

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add parent directory to path if running from examples folder
sys.path.insert(0, str(Path.cwd().parent))

from OSM_auto_label import (
    OSMLandcoverClassifier,
    LandcoverMapVisualizer,
    config,
)

print("OSM Auto Label loaded successfully!")
print(f"\nAvailable categories: {list(config.SEED_CATEGORIES.keys())}")

## Step 1: Classification

Initialize the classifier and run the processing pipeline:

In [ ]:
# Initialize classifier (this loads the word embedding model)
classifier = OSMLandcoverClassifier()

# Run classification pipeline
gdf_classified = classifier.process(
    input_shapefile=INPUT_SHAPEFILE,
    output_shapefile=OUTPUT_SHAPEFILE,
    run_clustering=RUN_CLUSTERING,
)

print(f"\nClassified {len(gdf_classified)} features")

## Step 2: Explore Results

Examine the classified data:

In [ ]:
# View sample of classified data
print("Sample of classified features:")
display(gdf_classified[['landuse', 'classname', 'classvalue', 'priority']].head(10))

# Category distribution
print("\nCategory distribution:")
display(gdf_classified['classname'].value_counts())

In [ ]:
# View which tags mapped to which categories
print("Tags per category:")
for cat, tags in classifier.cluster_assignments.items():
    print(f"\n{cat}:")
    print(f"  {', '.join(sorted(tags)[:15])}{'...' if len(tags) > 15 else ''}")

## Step 3: Visualization

Create an interactive map:

In [ ]:
# Initialize visualizer
visualizer = LandcoverMapVisualizer()

# Create and save map
m = visualizer.create_landcover_map(
    gdf_classified,
    output_path=OUTPUT_MAP,
    show_category_layers=False,  # Set True for separate toggleable layers
)

# Display map inline (in Jupyter)
m

### Alternative: Category Layers View

Create a map with separate toggleable layers for each category:

In [ ]:
# Create map with category layers
m_layers = visualizer.create_landcover_map(
    gdf_classified,
    output_path="output/landcover_map_layers.html",
    show_category_layers=True,
)

m_layers

## Advanced: Custom Configuration

You can override default categories, colors, and priorities:

In [ ]:
# Example: Custom seed categories
custom_categories = {
    "urban": ["residential", "commercial", "industrial", "retail"],
    "vegetation": ["forest", "grass", "meadow", "orchard"],
    "water": ["water", "wetland", "reservoir"],
    "agriculture": ["farmland", "allotments", "vineyard"],
}

custom_colors = {
    "urban": "#FF6B6B",
    "vegetation": "#4ECDC4",
    "water": "#45B7D1",
    "agriculture": "#96CEB4",
}

# Initialize with custom config
custom_classifier = OSMLandcoverClassifier(
    seed_categories=custom_categories,
)

custom_visualizer = LandcoverMapVisualizer(
    colors=custom_colors,
)

print("Custom classifier initialized with categories:", list(custom_categories.keys()))

## Advanced: Step-by-Step Processing

For more control, you can run each step individually:

In [ ]:
# Step-by-step processing for more control
classifier = OSMLandcoverClassifier()

# 1. Load data
gdf = classifier.load_shapefile(INPUT_SHAPEFILE)

# 2. Preprocess
gdf = classifier.preprocess(gdf)

# 3. Get tag statistics
tag_counts_list, tag_set = classifier.get_tag_statistics(gdf)
tag_counts = dict(tag_counts_list)

# 4. Create word vector mapping
landuse_key_map = classifier.create_landuse_mapping(tag_set)

# 5. Filter and assign categories
tag_counts_filtered = {k: v for k, v in tag_counts.items() if k in landuse_key_map}
cluster_assignments, cluster_assignments_rev = classifier.assign_categories(
    tag_counts_filtered, landuse_key_map
)

# 6. Classify GeoDataFrame
gdf_classified = classifier.classify(gdf, cluster_assignments_rev)

# 7. Save (optional)
# classifier.save_shapefile(gdf_classified, OUTPUT_SHAPEFILE)

print("Step-by-step processing complete!")